In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Context Graph V5: TTL Import, Mixed Extraction, Temporal Lineage

**Extending the BigQuery Agent Analytics SDK with OWL Import, Structured Extractors, and Cross-Session Lineage**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

## Overview

The **Context Graph V5** pipeline extends V4 with three new capabilities:

1. **OWL/Turtle Import** -- A two-phase importer (`ttl_import` + `ttl_resolve`) that parses
   OWL/Turtle ontology files via `rdflib`, maps OWL classes and properties to the GraphSpec
   YAML format, and produces a clean spec with `FILL_IN` placeholders resolved by the user.
   This bridges existing ontology standards (OWL 2, RDFS) into the configuration-driven pipeline.

2. **Structured Extraction** -- A typed extractor registry that converts known event types
   (e.g., `BKA_DECISION`) into `ExtractedNode`/`ExtractedEdge` instances deterministically,
   without AI. Extractors declare which spans they fully or partially handle, so downstream
   `AI.GENERATE` can skip already-covered data. This hybrid approach combines the precision of
   structured parsing with the flexibility of AI extraction.

3. **Temporal Lineage** -- Cross-session entity evolution detection via `detect_lineage_edges`.
   When the same entity (matched by primary key) appears in multiple sessions with different
   property values, a lineage edge is created to track how it changed over time. A dedicated
   `compile_lineage_gql` helper generates GQL queries for traversing these evolution paths.

Together, these features let you import an external ontology, extract graphs with mixed
structured+AI pipelines, and track how entities evolve across sessions -- all within the
same configuration-driven framework established in V4.

## Install Dependencies

In [ ]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery pyyaml rdflib

## Authenticate & Configure

In [ ]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab -- using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
ENV = f"{PROJECT_ID}.{DATASET_ID}"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")
print(f"Env      : {ENV}")

## Step 1: OWL/Turtle Import

The V5 pipeline starts with an optional **two-phase OWL/Turtle import** that bridges existing
ontology standards into the GraphSpec YAML format.

- **Phase 1 (`ttl_import`)**: Parses the `.ttl` file via `rdflib`, maps `owl:Class` to entities,
  `owl:DatatypeProperty` to typed properties, and `owl:ObjectProperty` to relationships. Where
  the OWL source lacks information (e.g., no `owl:hasKey`), the importer inserts `FILL_IN`
  placeholders and records each one in an `ImportReport`.

- **Phase 2 (`ttl_resolve`)**: Reads the import artifact, substitutes `FILL_IN` placeholders
  with user-supplied defaults, strips the `ontology_import:` metadata block, validates the
  result against `load_graph_spec_from_string`, and returns clean GraphSpec YAML.

In [ ]:
# Write the YAMO sample OWL/Turtle ontology into the working directory
# so it is available regardless of how the notebook was launched.

TTL_PATH = "yamo_sample.ttl"

_YAMO_TTL = """\
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix yamo: <https://example.com/yamo#> .

# ============================================================
# Classes
# ============================================================

yamo:Party a owl:Class ;
  rdfs:label "Party" ;
  owl:hasKey ( yamo:party_id ) .

yamo:AdUnit a owl:Class ;
  rdfs:label "Ad Unit" ;
  rdfs:subClassOf yamo:Party ;
  owl:hasKey ( yamo:ad_unit_id ) .

yamo:Campaign a owl:Class ;
  rdfs:label "Campaign" ;
  owl:hasKey ( yamo:campaign_id ) .

yamo:RejectionReason a owl:Class ;
  rdfs:label "Rejection Reason" ;
  owl:hasKey ( yamo:rejection_type ) .

# DecisionPoint has NO owl:hasKey -> should produce FILL_IN.
yamo:DecisionPoint a owl:Class ;
  rdfs:label "Decision Point" .

# ============================================================
# Datatype Properties
# ============================================================

yamo:party_id a owl:DatatypeProperty ;
  rdfs:domain yamo:Party ;
  rdfs:range xsd:string .

yamo:name a owl:DatatypeProperty ;
  rdfs:domain yamo:Party ;
  rdfs:range xsd:string .

yamo:ad_unit_id a owl:DatatypeProperty ;
  rdfs:domain yamo:AdUnit ;
  rdfs:range xsd:string .

yamo:campaign_id a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:string .

yamo:decision_id a owl:DatatypeProperty ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range xsd:string .

yamo:decision_type a owl:DatatypeProperty ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range xsd:string .

yamo:rejection_type a owl:DatatypeProperty ;
  rdfs:domain yamo:RejectionReason ;
  rdfs:range xsd:string .

# budget is xsd:decimal -> should narrow to double with a warning.
yamo:budget a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:decimal .

yamo:start_date a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:date .

# ============================================================
# Object Properties (relationships)
# ============================================================

yamo:evaluates a owl:ObjectProperty ;
  rdfs:label "evaluates" ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range yamo:AdUnit .

yamo:rejectedBy a owl:ObjectProperty ;
  rdfs:label "rejected by" ;
  rdfs:domain yamo:AdUnit ;
  rdfs:range yamo:RejectionReason .
"""

with open(TTL_PATH, "w") as _f:
    _f.write(_YAMO_TTL)

print(f"Wrote {TTL_PATH} ({len(_YAMO_TTL)} bytes)")

In [ ]:
from bigquery_agent_analytics.ttl_importer import ttl_import, ttl_resolve

result = ttl_import("yamo_sample.ttl", include_namespaces=["https://example.com/yamo#"])
print("=== Import Report ===")
print(f"Classes mapped: {result.report.classes_mapped}")
print(f"Properties mapped: {result.report.properties_mapped}")
print(f"Relationships mapped: {result.report.relationships_mapped}")
print(f"\nPlaceholders ({len(result.report.placeholders)}):")
for p in result.report.placeholders:
    print(f"  - {p.location}: {p.reason}")
print(f"\nType warnings ({len(result.report.type_warnings)}):")
for w in result.report.type_warnings:
    print(f"  - {w.property_name}: {w.owl_type} -> {w.mapped_type}")

In [ ]:
# Show the unresolved YAML artifact (first 40 lines)
print("=== Unresolved Import YAML (first 40 lines) ===")
for i, line in enumerate(result.yaml_text.splitlines()[:40]):
    print(line)

### Resolve Placeholders

Phase 2 (`ttl_resolve`) takes the import artifact and replaces `FILL_IN` placeholders with
user-supplied values. In this example, the `DecisionPoint` class had no `owl:hasKey`, so we
supply its primary key and the `evaluates` relationship's `from_columns` binding.

In [ ]:
import tempfile, os
from bigquery_agent_analytics.ontology_models import load_graph_spec_from_string

with tempfile.NamedTemporaryFile(mode="w", suffix=".import.yaml", delete=False) as f:
    f.write(result.yaml_text)
    import_path = f.name

resolved_yaml = ttl_resolve(import_path, defaults={
    "entities[DecisionPoint].keys.primary": ["decision_id"],
    "relationships[evaluates].binding.from_columns": ["decision_id"],
})
os.unlink(import_path)

spec_from_owl = load_graph_spec_from_string(resolved_yaml)
print(f"Resolved spec: {spec_from_owl.name}")
print(f"Entities: {[e.name for e in spec_from_owl.entities]}")
print(f"Relationships: {[r.name for r in spec_from_owl.relationships]}")

## Step 2: Mixed Structured + AI Extraction

V5 introduces a **structured extractor registry** that runs alongside AI extraction. Each
extractor is a function that takes a raw telemetry event and the `GraphSpec`, and returns
a `StructuredExtractionResult` with:

- **nodes/edges**: Deterministically extracted graph elements (no AI needed).
- **fully_handled_span_ids**: Spans whose content is completely captured -- excluded from
  the AI transcript to avoid duplicate extraction.
- **partially_handled_span_ids**: Spans with remaining free-text that AI should still process.

The `OntologyGraphManager` accepts an `extractors` dict mapping `event_type` strings to
extractor callables. During extraction, matched events are processed structurally first,
and only unhandled spans are sent to `AI.GENERATE`.

In [ ]:
# Write the YMGO graph spec with the V5 lineage relationship appended.

SPEC_PATH = "ymgo_graph_spec.yaml"

_YMGO_SPEC = """\
# YMGO (Yahoo Media Graph Ontology) V3 — Context Graph Specification
#
# This YAML defines the ontology for a configuration-driven context graph.
# Use {{ env }} placeholders for environment-specific table bindings,
# resolved at load time via load_graph_spec(path, env="project.dataset").

graph:
  name: YMGO_Context_Graph_V3

  # ==========================================
  # 1. ENTITIES (Nodes)
  # ==========================================
  entities:
    - name: mako_DecisionPoint
      description: "The atomic unit of decisioning where an agent evaluates alternatives."
      binding:
        source: "{{ env }}.decision_points"
      keys:
        primary: [decision_id]
      properties:
        - name: decision_id
          type: string
        - name: decision_type
          type: string

    - name: sup_YahooAdUnit
      extends: mako_Candidate
      description: "A specific ad slot on a Yahoo property being evaluated as a candidate."
      binding:
        source: "{{ env }}.yahoo_ad_units"
      keys:
        primary: [adUnitId]
      properties:
        - name: adUnitId
          type: string
        - name: adUnitName
          type: string
        - name: adUnitSize
          type: string
          description: "e.g., '300x250', '728x90'"
        - name: adUnitPosition
          type: string
          description: "ATF (above the fold) | BTF (below the fold)"

    - name: mako_RejectionReason
      description: "Structured reason why a Candidate was not selected at a DecisionPoint."
      binding:
        source: "{{ env }}.rejection_reasons"
      keys:
        primary: [rejection_id]
      properties:
        - name: rejection_id
          type: string
        - name: rejectionType
          type: string
          description: "RULE_BASED | MODEL_BASED | TIMEOUT | ERROR"
        - name: rejectionRule
          type: string
          description: "The specific rule or model threshold that caused rejection."

  # ==========================================
  # 2. RELATIONSHIPS (Edges)
  # ==========================================
  relationships:
    - name: CandidateEdge
      description: "Connects a decision point to the evaluated Yahoo Ad Unit."
      from_entity: mako_DecisionPoint
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.candidate_edges"
        from_columns: [decision_id]
        to_columns: [adUnitId]
      properties:
        - name: edge_type
          type: string
          description: "SELECTED_CANDIDATE or DROPPED_CANDIDATE"
        - name: mako_scoreValue
          type: double
          description: "The confidence or predicted Q-value for this candidate."

    - name: ForCandidate
      description: "Maps the MAKO rejection rationale directly to the dropped candidate."
      from_entity: mako_RejectionReason
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.rejection_mappings"
        from_columns: [rejection_id]
        to_columns: [adUnitId]

    - name: sup_YahooAdUnitEvolvedFrom
      description: "Tracks evolution of a YahooAdUnit across sessions."
      from_entity: sup_YahooAdUnit
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.sup_yahoo_ad_unit_lineage"
        from_columns: [adUnitId]
        to_columns: [adUnitId]
        from_session_column: from_session_id
        to_session_column: to_session_id
      properties:
        - name: from_session_id
          type: string
        - name: to_session_id
          type: string
        - name: event_time
          type: timestamp
        - name: changed_properties
          type: string
"""

with open(SPEC_PATH, "w") as _f:
    _f.write(_YMGO_SPEC)

print(f"Wrote {SPEC_PATH} ({len(_YMGO_SPEC)} bytes)")

from bigquery_agent_analytics.ontology_models import load_graph_spec

spec = load_graph_spec(SPEC_PATH, env=ENV)
print(f"Graph: {spec.name}")
print(f"Entities: {[e.name for e in spec.entities]}")
print(f"Relationships: {[r.name for r in spec.relationships]}")

In [ ]:
from bigquery_agent_analytics.structured_extraction import (
    extract_bka_decision_event,
    StructuredExtractionResult,
)

# Demo: run the extractor on a sample event
sample_event = {
    "span_id": "span-001",
    "session_id": "demo-session-1",
    "event_type": "BKA_DECISION",
    "content": {
        "decision_id": "dp_homepage_eval",
        "decision_type": "query_ad_inventory",
        "outcome": "selected",
    },
}
demo_result = extract_bka_decision_event(sample_event, spec)
print(f"Nodes extracted: {len(demo_result.nodes)}")
print(f"Fully handled spans: {demo_result.fully_handled_span_ids}")
for node in demo_result.nodes:
    print(f"  [{node.entity_name}] {node.node_id}")
    for p in node.properties:
        print(f"    {p.name} = {p.value}")

In [ ]:
from bigquery_agent_analytics.ontology_graph import OntologyGraphManager

SESSION_IDS = ["adcp-033c95d7a97d"]

extractor = OntologyGraphManager(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    spec=spec,
    table_id=TABLE_ID,
    endpoint="gemini-2.5-flash",
    extractors={"BKA_DECISION": extract_bka_decision_event},
)

graph = extractor.extract_graph(session_ids=SESSION_IDS, use_ai_generate=True)
print(f"Extracted {len(graph.nodes)} nodes, {len(graph.edges)} edges")
for node in graph.nodes[:5]:
    print(f"  [{node.entity_name}] {node.node_id}")

## Step 3: Materialize & Create Property Graph

With the extracted graph in hand, we create physical BigQuery tables, materialize the nodes
and edges, generate the Property Graph DDL (now including the lineage edge table), and create
the native BigQuery Property Graph.

In [ ]:
from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer
from bigquery_agent_analytics.ontology_property_graph import (
    OntologyPropertyGraphCompiler,
    compile_property_graph_ddl,
)

materializer = OntologyMaterializer(project_id=PROJECT_ID, dataset_id=DATASET_ID, spec=spec)
tables = materializer.create_tables()
print("Tables created:")
for name, ref in tables.items():
    print(f"  {name} -> {ref}")

rows = materializer.materialize(graph, SESSION_IDS)
print(f"\nRows materialized: {rows}")

In [ ]:
ddl = compile_property_graph_ddl(spec, PROJECT_ID, DATASET_ID)
print(ddl)

In [ ]:
compiler = OntologyPropertyGraphCompiler(project_id=PROJECT_ID, dataset_id=DATASET_ID, spec=spec)
created = compiler.create_property_graph()
print(f"Property Graph created: {created}")

## Step 4: Temporal Lineage -- Cross-Session Entity Evolution

V5 introduces **temporal lineage**: the ability to detect how entities evolve across sessions.
When the same entity (matched by primary key) appears in two sessions with different property
values, `detect_lineage_edges` creates a lineage edge recording:

- The prior and current session IDs
- A timestamp of when the evolution was detected
- A comma-separated list of properties that changed

This enables use cases like tracking how ad unit configurations drift over time, or auditing
which decision parameters changed between sessions.

In [ ]:
SESSION_IDS_B = ["adcp-040c04837251"]  # Second session

graph_b = extractor.extract_graph(session_ids=SESSION_IDS_B, use_ai_generate=True)
print(f"Session B: {len(graph_b.nodes)} nodes, {len(graph_b.edges)} edges")

# Materialize session B
rows_b = materializer.materialize(graph_b, SESSION_IDS_B)
print(f"Session B rows: {rows_b}")

In [ ]:
from bigquery_agent_analytics.ontology_graph import detect_lineage_edges

lineage_edges = detect_lineage_edges(
    current_graph=graph_b,
    current_session_id=SESSION_IDS_B[0],
    prior_graphs={SESSION_IDS[0]: graph},
    lineage_entity_types=["sup_YahooAdUnit"],
    spec=spec,
)

print(f"Lineage edges detected: {len(lineage_edges)}")
for edge in lineage_edges:
    print(f"\n  {edge.relationship_name}: {edge.from_node_id} -> {edge.to_node_id}")
    for p in edge.properties:
        print(f"    {p.name} = {p.value}")

## Step 5: GQL Queries

V5 adds `compile_lineage_gql` alongside the existing V4 `compile_showcase_gql`. The lineage
GQL traverses self-edge relationships (where `from_entity == to_entity`) and returns prior
and current session property values with the evolution metadata.

In [ ]:
from bigquery_agent_analytics.ontology_orchestrator import compile_showcase_gql, compile_lineage_gql

print("=== V4: Forward Traversal GQL ===")
gql = compile_showcase_gql(spec, PROJECT_ID, DATASET_ID)
print(gql)

print("\n=== V5: Cross-Session Lineage GQL ===")
lineage_gql = compile_lineage_gql(
    spec=spec,
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    relationship_name="sup_YahooAdUnitEvolvedFrom",
)
print(lineage_gql)

In [ ]:
from google.cloud import bigquery as bq_client_mod

bq = bq_client_mod.Client(project=PROJECT_ID)
job = bq.query(gql, job_config=bq_client_mod.QueryJobConfig(
    query_parameters=[
        bq_client_mod.ScalarQueryParameter("session_id", "STRING", SESSION_IDS[0]),
        bq_client_mod.ScalarQueryParameter("result_limit", "INT64", 50),
    ]
))
results = job.to_dataframe()
print(f"Forward traversal: {len(results)} rows")
results.head(10)

## Summary

The V5 Context Graph pipeline extends V4 with three new capabilities:

| Feature | V4 | V5 |
|---------|----|----|
| **Ontology Source** | Hand-authored YAML only | YAML + OWL/Turtle import via `ttl_import`/`ttl_resolve` |
| **Extraction** | AI-only (`AI.GENERATE`) | Mixed: structured extractors + AI for unhandled spans |
| **Scope** | Single-session graph | Cross-session temporal lineage via `detect_lineage_edges` |
| **GQL** | `compile_showcase_gql` (forward traversal) | + `compile_lineage_gql` (self-edge evolution queries) |
| **Spec Format** | Entities + relationships | + lineage self-edge relationships with session columns |

**Key APIs introduced in V5:**

- `bigquery_agent_analytics.ttl_importer.ttl_import` -- Phase 1 OWL-to-YAML import
- `bigquery_agent_analytics.ttl_importer.ttl_resolve` -- Phase 2 placeholder resolution
- `bigquery_agent_analytics.structured_extraction.extract_bka_decision_event` -- Example structured extractor
- `bigquery_agent_analytics.structured_extraction.run_structured_extractors` -- Extractor runner
- `bigquery_agent_analytics.ontology_graph.detect_lineage_edges` -- Cross-session evolution detection
- `bigquery_agent_analytics.ontology_orchestrator.compile_lineage_gql` -- Lineage GQL compiler
- `OntologyGraphManager(extractors=...)` -- New parameter to register structured extractors